# Chapter 8 (Modeling) -- Results Notebook

Re-runs the real hyperparameter and blend-weight searches (or loads their saved results, if already persisted) and prints:
1. Coxnet l1_ratio search results, both endpoints -- as LaTeX table rows, ready to paste.
2. XGBoost hyperparameter search results, both endpoints -- as LaTeX table rows.
3. Blend weight search results (winning weights), both endpoints -- as LaTeX table rows.
4. A bar chart: individual model C-index vs. final blend, per endpoint.

Run this top to bottom, then copy the printed LaTeX snippets directly into the chapter tables, and send me the printed numbers/chart so I can write the surrounding text accurately.

In [1]:
import sys
from pathlib import Path

def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "liverrisk" / "features.py").exists():
            return p
    raise RuntimeError("Could not locate repo root (liverrisk/features.py not found)")

REPO_ROOT = _find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("REPO_ROOT:", REPO_ROOT)

REPO_ROOT: c:\Users\paabl\OneDrive\Documents\GitHub\TFG_pabloCalderon


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from liverrisk import config
from liverrisk.features import load_features
from liverrisk.models import search_coxnet_l1_ratio, search_xgb_hyperparams
from liverrisk.cv import cv_cindex_coxnet, cv_cindex_rsf, cv_cindex_xgb, cv_cindex_blend
from liverrisk.blend import search_blend_weights

PROCESSED_DIR = REPO_ROOT / "liverrisk" / "data" / "processed"

X_hep, y_hep, hep_event, hep_time = load_features("hep", PROCESSED_DIR)
X_death, y_death, death_event, death_time = load_features("death", PROCESSED_DIR)

print(f"Hepatic cohort: {len(X_hep)} patients, {hep_event.sum()} events")
print(f"Death cohort: {len(X_death)} patients, {death_event.sum()} events")

Hepatic cohort: 1253 patients, 47 events
Death cohort: 984 patients, 76 events


## 1. Coxnet l1_ratio search results (both endpoints)

In [3]:
print("Searching Coxnet l1_ratio -- hepatic...")
best_l1_hep, best_mean_hep, results_hep_l1 = search_coxnet_l1_ratio(X_hep, y_hep, hep_event, hep_time, n_repeats=1)
print(f"Best hepatic l1_ratio = {best_l1_hep}, mean C-index = {best_mean_hep:.4f}")
print(results_hep_l1)

print("\nSearching Coxnet l1_ratio -- death...")
best_l1_death, best_mean_death, results_death_l1 = search_coxnet_l1_ratio(X_death, y_death, death_event, death_time, n_repeats=1)
print(f"Best death l1_ratio = {best_l1_death}, mean C-index = {best_mean_death:.4f}")
print(results_death_l1)

Searching Coxnet l1_ratio -- hepatic...
Best hepatic l1_ratio = 0.1, mean C-index = 0.7665
   l1_ratio      mean       std
0      0.10  0.766536  0.110555
1      0.30  0.731945  0.102697
2      0.50  0.729350  0.106411
3      0.70  0.723619  0.107280
4      0.90  0.719686  0.105788
5      0.95  0.719019  0.104115
6      0.99  0.718142  0.102255

Searching Coxnet l1_ratio -- death...
Best death l1_ratio = 0.99, mean C-index = 0.9404
   l1_ratio      mean       std
0      0.99  0.940378  0.016437
1      0.95  0.939615  0.016620
2      0.90  0.937279  0.017335
3      0.50  0.933204  0.020698
4      0.70  0.933111  0.017335
5      0.10  0.930301  0.021224
6      0.30  0.928702  0.018980


In [4]:
# Print as LaTeX table rows, ready to paste
print("=== LaTeX rows: Coxnet l1_ratio search (hepatic) ===")
for _, row in results_hep_l1.iterrows():
    print(f"{row['l1_ratio']} & {row['mean']:.4f} & {row['std']:.4f} \\\\")

print("\n=== LaTeX rows: Coxnet l1_ratio search (death) ===")
for _, row in results_death_l1.iterrows():
    print(f"{row['l1_ratio']} & {row['mean']:.4f} & {row['std']:.4f} \\\\")

=== LaTeX rows: Coxnet l1_ratio search (hepatic) ===
0.1 & 0.7665 & 0.1106 \\
0.3 & 0.7319 & 0.1027 \\
0.5 & 0.7293 & 0.1064 \\
0.7 & 0.7236 & 0.1073 \\
0.9 & 0.7197 & 0.1058 \\
0.95 & 0.7190 & 0.1041 \\
0.99 & 0.7181 & 0.1023 \\

=== LaTeX rows: Coxnet l1_ratio search (death) ===
0.99 & 0.9404 & 0.0164 \\
0.95 & 0.9396 & 0.0166 \\
0.9 & 0.9373 & 0.0173 \\
0.5 & 0.9332 & 0.0207 \\
0.7 & 0.9331 & 0.0173 \\
0.1 & 0.9303 & 0.0212 \\
0.3 & 0.9287 & 0.0190 \\


## 2. XGBoost hyperparameter search results (both endpoints)

In [6]:
print("Searching XGBoost hyperparameters -- hepatic...")
best_xgb_hep, best_xgb_mean_hep, results_xgb_hep = search_xgb_hyperparams(X_hep, y_hep, hep_event, hep_time, n_candidates=12)
print(f"Best hepatic XGB params: {best_xgb_hep}")
print(f"Mean C-index: {best_xgb_mean_hep:.4f}")

print("\nSearching XGBoost hyperparameters -- death...")
best_xgb_death, best_xgb_mean_death, results_xgb_death = search_xgb_hyperparams(X_death, y_death, death_event, death_time, n_candidates=12)
print(f"Best death XGB params: {best_xgb_death}")
print(f"Mean C-index: {best_xgb_mean_death:.4f}")

Searching XGBoost hyperparameters -- hepatic...
Best hepatic XGB params: {'learning_rate': 0.025, 'max_depth': 3, 'n_estimators': 300, 'min_child_weight': 5, 'subsample': 0.7, 'colsample_bytree': 0.7, 'reg_lambda': 5.0, 'reg_alpha': 0.5}
Mean C-index: 0.7402

Searching XGBoost hyperparameters -- death...
Best death XGB params: {'learning_rate': 0.1, 'max_depth': 2, 'n_estimators': 300, 'min_child_weight': 5, 'subsample': 0.7, 'colsample_bytree': 0.85, 'reg_lambda': 1.0, 'reg_alpha': 0.5}
Mean C-index: 0.9366


In [7]:
# Print as LaTeX table rows -- winning config only, one row per endpoint
print("=== LaTeX rows: XGBoost winning hyperparameters ===")
for label, params, mean in [("Hepatic", best_xgb_hep, best_xgb_mean_hep), ("Death", best_xgb_death, best_xgb_mean_death)]:
    print(f"{label} & {params['learning_rate']} & {params['max_depth']} & {params['n_estimators']} & "
          f"{params['min_child_weight']} & {params['subsample']} & {params['colsample_bytree']} & "
          f"{params['reg_lambda']} & {params['reg_alpha']} & {mean:.4f} \\\\")

=== LaTeX rows: XGBoost winning hyperparameters ===
Hepatic & 0.025 & 3 & 300 & 5 & 0.7 & 0.7 & 5.0 & 0.5 & 0.7402 \\
Death & 0.1 & 2 & 300 & 5 & 0.7 & 0.85 & 1.0 & 0.5 & 0.9366 \\


## 3. Blend weight search results (both endpoints)

In [8]:
print("Searching blend weights -- hepatic...")
best_weights_hep, best_blend_mean_hep, results_blend_hep = search_blend_weights(
    X_hep, y_hep, hep_event, hep_time, n_points=6, xgb_params=best_xgb_hep
)
print(f"Best hepatic weights (cox, rsf, xgb) = {best_weights_hep}, mean C-index = {best_blend_mean_hep:.4f}")
print(results_blend_hep.head(10))

print("\nSearching blend weights -- death...")
best_weights_death, best_blend_mean_death, results_blend_death = search_blend_weights(
    X_death, y_death, death_event, death_time, n_points=6, xgb_params=best_xgb_death
)
print(f"Best death weights (cox, rsf, xgb) = {best_weights_death}, mean C-index = {best_blend_mean_death:.4f}")
print(results_blend_death.head(10))

Searching blend weights -- hepatic...


KeyboardInterrupt: 

In [ ]:
# Print as LaTeX table rows
print("=== LaTeX rows: Blend weight search winners ===")
print(f"Hepatic & {best_weights_hep[0]} & {best_weights_hep[1]} & {best_weights_hep[2]} & {best_blend_mean_hep:.4f} \\\\")
print(f"Death & {best_weights_death[0]} & {best_weights_death[1]} & {best_weights_death[2]} & {best_blend_mean_death:.4f} \\\\")

## 4. Final trustworthy evaluation (n_repeats=3) and chart: individual models vs. blend

In [ ]:
print("Running final n_repeats=3 evaluation -- this may take a few minutes...")

cox_hep_mean, cox_hep_std = cv_cindex_coxnet(X_hep, y_hep, hep_event, n_repeats=3, l1_ratio=best_l1_hep)
rsf_hep_mean, rsf_hep_std = cv_cindex_rsf(X_hep, y_hep, hep_event, n_repeats=3)
xgb_hep_mean, xgb_hep_std = cv_cindex_xgb(X_hep, hep_event, hep_time, n_repeats=3, xgb_params=best_xgb_hep)
blend_hep_mean, blend_hep_std = cv_cindex_blend(X_hep, y_hep, hep_event, hep_time, n_repeats=3, weights=list(best_weights_hep), xgb_params=best_xgb_hep)

cox_death_mean, cox_death_std = cv_cindex_coxnet(X_death, y_death, death_event, n_repeats=3, l1_ratio=best_l1_death)
rsf_death_mean, rsf_death_std = cv_cindex_rsf(X_death, y_death, death_event, n_repeats=3)
xgb_death_mean, xgb_death_std = cv_cindex_xgb(X_death, death_event, death_time, n_repeats=3, xgb_params=best_xgb_death)
blend_death_mean, blend_death_std = cv_cindex_blend(X_death, y_death, death_event, death_time, n_repeats=3, weights=list(best_weights_death), xgb_params=best_xgb_death)

print("\n=== FINAL RESULTS (n_repeats=3) ===")
print(f"Hepatic -- Coxnet: {cox_hep_mean:.4f}+-{cox_hep_std:.4f}, RSF: {rsf_hep_mean:.4f}+-{rsf_hep_std:.4f}, "
      f"XGB: {xgb_hep_mean:.4f}+-{xgb_hep_std:.4f}, Blend: {blend_hep_mean:.4f}+-{blend_hep_std:.4f}")
print(f"Death   -- Coxnet: {cox_death_mean:.4f}+-{cox_death_std:.4f}, RSF: {rsf_death_mean:.4f}+-{rsf_death_std:.4f}, "
      f"XGB: {xgb_death_mean:.4f}+-{xgb_death_std:.4f}, Blend: {blend_death_mean:.4f}+-{blend_death_std:.4f}")

weighted_score = 0.7 * blend_hep_mean + 0.3 * blend_death_mean
print(f"\nOfficial weighted score (0.7*hep + 0.3*death): {weighted_score:.4f}")

In [ ]:
# LaTeX table rows for the final results table
print("=== LaTeX rows: Final individual model + blend results ===")
print(f"Coxnet & {cox_hep_mean:.4f} $\\pm$ {cox_hep_std:.4f} & {cox_death_mean:.4f} $\\pm$ {cox_death_std:.4f} \\\\")
print(f"Random Survival Forest & {rsf_hep_mean:.4f} $\\pm$ {rsf_hep_std:.4f} & {rsf_death_mean:.4f} $\\pm$ {rsf_death_std:.4f} \\\\")
print(f"XGBoost & {xgb_hep_mean:.4f} $\\pm$ {xgb_hep_std:.4f} & {xgb_death_mean:.4f} $\\pm$ {xgb_death_std:.4f} \\\\")
print(f"\\textbf{{Blended Ensemble}} & \\textbf{{{blend_hep_mean:.4f} $\\pm$ {blend_hep_std:.4f}}} & \\textbf{{{blend_death_mean:.4f} $\\pm$ {blend_death_std:.4f}}} \\\\")
print(f"\n\\textbf{{Official weighted score: {weighted_score:.4f}}}")

In [ ]:
# Bar chart: individual models vs. blend, per endpoint
fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharey=True)

labels = ["Coxnet", "RSF", "XGBoost", "Blend"]
hep_vals = [cox_hep_mean, rsf_hep_mean, xgb_hep_mean, blend_hep_mean]
hep_errs = [cox_hep_std, rsf_hep_std, xgb_hep_std, blend_hep_std]
death_vals = [cox_death_mean, rsf_death_mean, xgb_death_mean, blend_death_mean]
death_errs = [cox_death_std, rsf_death_std, xgb_death_std, blend_death_std]

colors = ["#8FA6A0", "#8FA6A0", "#8FA6A0", "#1F6F5C"]

axes[0].bar(labels, hep_vals, yerr=hep_errs, color=colors, capsize=4)
axes[0].set_title("Hepatic event")
axes[0].set_ylabel("C-index")
axes[0].set_ylim(0.5, 1.0)

axes[1].bar(labels, death_vals, yerr=death_errs, color=colors, capsize=4)
axes[1].set_title("Death")
axes[1].set_ylim(0.5, 1.0)

plt.suptitle("Individual model vs. blended ensemble performance")
plt.tight_layout()

OUTPUT_DIR = REPO_ROOT / "chapter8_figures"
OUTPUT_DIR.mkdir(exist_ok=True)
plt.savefig(OUTPUT_DIR / "fig_model_vs_blend_performance.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"\nSaved to {OUTPUT_DIR / 'fig_model_vs_blend_performance.png'}")

## Done

Copy every printed LaTeX row block above into the corresponding chapter table, and send me:
1. All the printed numbers (search results, final results, weighted score)
2. The saved chart (`fig_model_vs_blend_performance.png`)
3. Confirmation of the 'Other/uncategorized'-style checks (nothing to check here, but flag anything that looks wrong or unexpected)

so I can write the surrounding chapter text using your real, verified numbers -- not placeholders.